# 🔬 ISIC 2020 – Melanoma Classification com PyTorch

**Objetivo:** Classificação binária de lesões de pele (benigno × maligno) usando o dataset ISIC 2020.  
**GPU alvo:** NVIDIA RTX 2080 Super (8 GB VRAM) — treinamento com AMP (Automatic Mixed Precision).  
**Backbone:** EfficientNet-B4 pré-treinado (ImageNet) via `timm`.

## Estrutura esperada dos dados
```
data/isic2020/
├── train/                              # imagens JPEG de treino
├── test/                               # imagens JPEG de teste
├── ISIC_2020_Training_GroundTruth.csv  # rótulos de treino
└── ISIC_2020_Test_Metadata.csv         # metadados de teste
```

## Pipeline
1. EDA dos CSVs
2. Dataset + Augmentations (albumentations)
3. Modelo EfficientNet-B4 com cabeça customizada
4. Treino com AMP, scheduler cosseno e early stopping
5. Avaliação: AUC-ROC, curva ROC, matriz de confusão
6. Inferência + `submission.csv`

## 0. Instalação de dependências

In [1]:
# Execute apenas se as bibliotecas não estiverem instaladas
import sys
!python.exe -m pip install --upgrade pip
!{sys.executable} -m pip install matplotlib
!{sys.executable} -m pip install scikit-learn
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install seaborn
!{sys.executable} -m pip install tqdm
!{sys.executable} -m pip install albumentations
!{sys.executable} -m pip install timm
!{sys.executable} -m pip install ipywidgets jupyter
!{sys.executable} -m pip install --upgrade tinycss2

print('instalação efetuada com sucesso!')

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2


  Using cached tinycss2-1.4.0-py3-none-any.whl.metadata (3.0 kB)
Using cached tinycss2-1.4.0-py3-none-any.whl (26 kB)
  Attempting uninstall: tinycss2
    Found existing installation: tinycss2 1.5.1
    Uninstalling tinycss2-1.5.1:
      Successfully uninstalled tinycss2-1.5.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
weasyprint 68.1 requires tinycss2>=1.5.0, but you have tinycss2 1.4.0 which is incompatible.


  Using cached tinycss2-1.5.1-py3-none-any.whl.metadata (3.0 kB)
Using cached tinycss2-1.5.1-py3-none-any.whl (28 kB)
  Attempting uninstall: tinycss2
    Found existing installation: tinycss2 1.4.0
    Uninstalling tinycss2-1.4.0:
      Successfully uninstalled tinycss2-1.4.0
instalação efetuada com sucesso!


## 1. Imports e configuração global

In [2]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

# ── Reproducibilidade ──────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cpu


In [3]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
print("Versão CUDA usada pelo PyTorch:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))


CUDA disponível: False
Versão CUDA usada pelo PyTorch: None


## 2. Configuração – caminhos e hiperparâmetros

In [ ]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path('../data/isic2020')        # raiz do dataset
TRAIN_DIR  = BASE_DIR / 'Train'              # imagens JPEG de treino
TEST_DIR   = BASE_DIR / 'test'               # imagens JPEG de teste
TRAIN_CSV  = BASE_DIR / 'ISIC_2020_Training_GroundTruth.csv'
TEST_CSV   = BASE_DIR / 'ISIC_2020_Test_Metadata.csv'
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Modelo ────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'efficientnet_b4'   # backbone timm
IMG_SIZE    = 380                  # resolução nativa do EfficientNet-B4
PRETRAINED  = True

# ── Treino ────────────────────────────────────────────────────────────────────
NUM_EPOCHS     = 15
BATCH_SIZE     = 16        # seguro para 8 GB VRAM com EfficientNet-B4 @ 380px
ACCUMULATE     = 2         # gradient accumulation → batch efetivo = 32
LR             = 1e-4
WEIGHT_DECAY   = 1e-5
N_FOLDS        = 5
FOLD_TO_TRAIN  = 0         # treine um fold; altere para iterar todos
NUM_WORKERS    = 0         # 0 = necessário no Windows/Jupyter (evita 'worker exited unexpectedly')
EARLY_STOPPING = 5         # paciência em épocas
USE_AMP        = True      # Automatic Mixed Precision (FP16 na RTX 2080 Super)

# ── Classes ───────────────────────────────────────────────────────────────────
TARGET_COL  = 'target'     # 0 = benigno, 1 = maligno
IMG_ID_COL  = 'image_name'
CLASS_NAMES = ['Benigno', 'Maligno']

# ── Metadados clínicos ────────────────────────────────────────────────────────
# Categorias de localização anatômica (ordem fixa → one-hot consistente)
ANATOM_SITE_CATS = [
    'torso', 'lower extremity', 'upper extremity',
    'head/neck', 'palms/soles', 'oral/genital',
]
# dim do vetor de meta: 1 (age_approx normalizada) + len(ANATOM_SITE_CATS) (one-hot)
NUM_META = 1 + len(ANATOM_SITE_CATS)   # = 7

print('Configuração carregada ✔')

## 3. EDA – Análise Exploratória

In [ ]:
import pandas as pd

# Carregando todos os registros do dataset
dataset_todos_train = pd.read_csv(TRAIN_CSV)
dataset_todos_teste = pd.read_csv(TEST_CSV)

# Filtrando somente os femininos e fazendo uma cópia
df_train = dataset_todos_train[dataset_todos_train['sex'] == 'female'].copy()
df_test  = dataset_todos_teste[dataset_todos_teste['sex'] == 'female'].copy()

print(f'Treino : {len(df_train):,} imagens')
print(f'Teste  : {len(df_test):,} imagens')
print('\nColunas treino:', df_train.columns.tolist())
print('Colunas teste :', df_test.columns.tolist())

# ── Pré-processamento de metadados clínicos ───────────────────────────────────
def preprocess_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza age_approx e adiciona colunas one-hot de anatom_site."""
    df = df.copy()
    
    # idade: preenche nulos com mediana e normaliza para [0, 1]
    age_median = df['age_approx'].median()
    df['age_norm'] = df['age_approx'].fillna(age_median) / 90.0
    
    # detectar coluna correta de anatomia
    if 'anatom_site_general_challenge' in df.columns:
        site_col = 'anatom_site_general_challenge'
    elif 'anatom_site_general' in df.columns:
        site_col = 'anatom_site_general'
    else:
        site_col = None
    
    # one-hot de localização anatômica (categorias fixas → compatível com API)
    for cat in ANATOM_SITE_CATS:
        col_name = f'site_{cat.replace("/", "_").replace(" ", "_")}'
        if site_col:
            df[col_name] = (df[site_col].fillna('') == cat).astype(float)
        else:
            df[col_name] = 0.0
    
    return df

df_train = preprocess_metadata(df_train)
df_test  = preprocess_metadata(df_test)

# colunas de meta na mesma ordem de ANATOM_SITE_CATS
META_COLS = ['age_norm'] + [
    f'site_{c.replace("/", "_").replace(" ", "_")}' for c in ANATOM_SITE_CATS
]

# renomear coluna de ID do teste para corresponder ao treino
if 'image' in df_test.columns and 'image_name' not in df_test.columns:
    df_test = df_test.rename(columns={'image': 'image_name'})

# alinhar colunas do teste com as do treino
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

print(f'Colunas de metadados ({len(META_COLS)}): {META_COLS}')
df_train.head()


In [ ]:
df_train.info()
print('\nValores nulos:')
print(df_train.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribuição de classes
counts = df_train[TARGET_COL].value_counts()
axes[0].bar(CLASS_NAMES, counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Distribuição de Classes')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df_train)*100:.1f}%)', ha='center')

# Distribuição por sexo
sns.countplot(data=df_train, x='sex', hue=TARGET_COL, ax=axes[1],
              palette=['steelblue', 'tomato'])
axes[1].set_title('Distribuição por Sexo')
axes[1].legend(CLASS_NAMES)

# Distribuição por localização anatômica
site_counts = df_train.groupby(['anatom_site_general_challenge', TARGET_COL]).size().unstack(fill_value=0)
site_counts.plot(kind='barh', ax=axes[2], color=['steelblue', 'tomato'])
axes[2].set_title('Distribuição por Localização Anatômica')
axes[2].legend(CLASS_NAMES)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualizar amostras de imagens de treino
sample_benign   = df_train[df_train[TARGET_COL] == 0].sample(4, random_state=SEED)
sample_malignant= df_train[df_train[TARGET_COL] == 1].sample(4, random_state=SEED)
samples = pd.concat([sample_benign, sample_malignant])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, row) in zip(axes.flatten(), samples.iterrows()):
    img_path = TRAIN_DIR / f"{row[IMG_ID_COL]}.jpg"
    if img_path.exists():
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
    label = CLASS_NAMES[row[TARGET_COL]]
    ax.set_title(f"{label}\n{row[IMG_ID_COL]}", fontsize=8)
    ax.axis('off')
plt.suptitle('Amostras de Imagens (top: Benigno | bottom: Maligno)', y=1.01)
plt.tight_layout()
plt.show()

## 4. Preparação dos Folds (Validação Cruzada Estratificada)

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df_train['fold'] = -1

# usamos .iloc porque val_idx são índices posicionais
for fold, (_, val_idx) in enumerate(skf.split(df_train, df_train[TARGET_COL])):
    df_train.iloc[val_idx, df_train.columns.get_loc('fold')] = fold

# ver distribuição de classes por fold
print(df_train.groupby(['fold', TARGET_COL]).size().unstack())



## 5. Dataset e Augmentations

In [ ]:
def get_transforms(phase: str, img_size: int = IMG_SIZE):
    if phase == 'train':
        return A.Compose([
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.7, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
            A.OneOf([
                A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0),
                A.GridDistortion(p=1.0),
                A.OpticalDistortion(p=1.0),
            ], p=0.3),
            A.OneOf([
                A.GaussNoise(p=1.0),
                A.GaussianBlur(p=1.0),
                A.MotionBlur(p=1.0),
            ], p=0.2),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ])
    else:  # val / test
        return A.Compose([
            A.Resize(height=img_size, width=img_size),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ])

print('Transforms definidos ✔')


In [ ]:
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform

        # Filtra apenas imagens existentes
        self.df = self.df[self.df[IMG_ID_COL].apply(
            lambda x: (self.img_dir / f"{x}.jpg").exists()
        )].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / f"{row[IMG_ID_COL]}.jpg"
        image = np.array(Image.open(img_path).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']

        # Converte meta para float
        meta = torch.tensor(row[META_COLS].astype(float).values, dtype=torch.float)
        label = torch.tensor(row[TARGET_COL], dtype=torch.float)

        return image, meta, label


print('ISICDataset definido ✔')

## 6. DataLoaders

In [ ]:
def build_loaders(df: pd.DataFrame, fold: int):
    df_tr  = df[df['fold'] != fold].reset_index(drop=True)
    df_val = df[df['fold'] == fold].reset_index(drop=True)

    # Cálculo de pos_weight para lidar com desbalanceamento
    n_neg    = (df_tr[TARGET_COL] == 0).sum()
    n_pos    = (df_tr[TARGET_COL] == 1).sum()
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
    print(f'Fold {fold} | treino={len(df_tr):,} | val={len(df_val):,} | '
          f'pos_weight={pos_weight.item():.2f}')

    ds_tr  = ISICDataset(df_tr,  TRAIN_DIR, transform=get_transforms('train'))
    ds_val = ISICDataset(df_val, TRAIN_DIR, transform=get_transforms('val'))

    loader_tr = DataLoader(
        ds_tr, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
    )
    loader_val = DataLoader(
        ds_val, batch_size=BATCH_SIZE * 2, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    return loader_tr, loader_val, pos_weight


loader_tr, loader_val, pos_weight = build_loaders(df_train, fold=FOLD_TO_TRAIN)
print(f'Batches treino: {len(loader_tr)} | val: {len(loader_val)}')

## 7. Modelo – EfficientNet-B4 com cabeça customizada

In [ ]:
class MelanomaClassifier(nn.Module):
    """EfficientNet-B4 multi-modal com metadados clínicos.

    Combina features visuais (backbone EfficientNet-B4) com metadados
    tabulares (age_approx normalizada + one-hot de anatom_site) por
    concatenação antes da cabeça de classificação.
    """

    def __init__(self, model_name: str = MODEL_NAME, pretrained: bool = PRETRAINED,
                 drop_rate: float = 0.3, num_meta: int = NUM_META):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0,   # remove a cabeça original
            drop_rate=drop_rate
        )
        in_features = self.backbone.num_features

        # Ramo de metadados: MLP pequeno
        self.meta_branch = nn.Sequential(
            nn.Linear(num_meta, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.2),
        )

        # Cabeça de classificação (img features + meta features)
        self.head = nn.Sequential(
            nn.Linear(in_features + 64, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1),   # saída logit única
        )

    def forward(self, x, meta):
        img_features  = self.backbone(x)        # (B, in_features)
        meta_features = self.meta_branch(meta)  # (B, 64)
        combined = torch.cat([img_features, meta_features], dim=1)
        return self.head(combined).squeeze(1)   # (B,)


model = MelanomaClassifier().to(DEVICE)

# Parâmetros totais e treináveis
total_params    = sum(p.numel() for p in model.parameters())
trainable_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros totais   : {total_params:,}')
print(f'Parâmetros treináveis: {trainable_params:,}')

## 8. Funções de treino e validação com AMP

In [ ]:
# imports necessários
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, scaler, accumulate=ACCUMULATE):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc='  train')
    for step, (images, meta, labels) in pbar:
        images = images.to(DEVICE, non_blocking=True)
        meta   = meta.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True).float()

        with autocast(enabled=USE_AMP):
            logits = model(images, meta)
            loss   = criterion(logits, labels) / accumulate

        scaler.scale(loss).backward()

        if (step + 1) % accumulate == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accumulate
        pbar.set_postfix({'loss': f'{running_loss / (step + 1):.4f}'})

    return running_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds  = []

    with torch.no_grad():
        pbar = tqdm(loader, total=len(loader), desc='  val ')
        for images, meta, labels in pbar:
            images = images.to(DEVICE, non_blocking=True)
            meta   = meta.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()

            logits = model(images, meta)
            loss   = criterion(logits, labels)

            preds = torch.sigmoid(logits)

            running_loss += loss.item()
            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

            pbar.set_postfix({'val_loss': f'{running_loss / (len(all_labels)):.4f}'})

    all_labels = np.concatenate(all_labels)
    all_preds  = np.concatenate(all_preds)

    # AUC binário
    try:
        val_auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        val_auc = 0.0

    return running_loss / len(loader), val_auc, all_preds, all_labels


## 9. Loop de Treinamento

In [ ]:
 # otimizador, scheduler, criterion, scaler (mantidos)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LR * 0.01)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = GradScaler(enabled=USE_AMP)

history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_auc = 0.0
patience_ct = 0
CKPT_PATH = OUTPUT_DIR / f'best_fold{FOLD_TO_TRAIN}.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}  (lr={scheduler.get_last_lr()[0]:.2e})')

    tr_loss = train_one_epoch(model, loader_tr, criterion, optimizer, scaler)
    val_loss, val_auc, val_preds, val_labels = validate(model, loader_val, criterion)

    # step do scheduler por epoch
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    print(f'  tr_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | val_AUC={val_auc:.4f}')

    if val_auc > best_auc:
        best_auc = val_auc
        patience_ct = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_auc': best_auc,
            'history': history
        }, CKPT_PATH)
        print(f'  ✔ Novo melhor AUC: {best_auc:.4f} → checkpoint salvo')
    else:
        patience_ct += 1
        print(f'  Sem melhora ({patience_ct}/{EARLY_STOPPING})')
        if patience_ct >= EARLY_STOPPING:
            print('  Early stopping ativado!')
            break

print(f'\nMelhor AUC de validação: {best_auc:.4f}')


## 10. Curvas de Aprendizado

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history['train_loss'], 'b-o', label='Treino')
ax1.plot(epochs_ran, history['val_loss'],   'r-o', label='Validação')
ax1.set_title('Loss por Época')
ax1.set_xlabel('Época')
ax1.set_ylabel('BCEWithLogitsLoss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_ran, history['val_auc'], 'g-o', label='Val AUC')
ax2.axhline(best_auc, color='orange', linestyle='--', label=f'Melhor AUC={best_auc:.4f}')
ax2.set_title('AUC-ROC por Época')
ax2.set_xlabel('Época')
ax2.set_ylabel('AUC-ROC')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Avaliação Final no Conjunto de Validação

In [ ]:
# Carregar melhor checkpoint
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
_, final_auc, y_true, y_prob = validate(model, loader_val, criterion)
print(f'AUC-ROC final (val fold {FOLD_TO_TRAIN}): {final_auc:.4f}')

In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, classification_report

# ── Curva ROC ────────────────────────────────────────────────────────────────
# garantir que y_true seja binário (0/1)
y_true = np.array(y_true).astype(int)

# se y_prob for matriz (ex: predict_proba), pegar coluna da classe positiva
if y_prob.ndim > 1 and y_prob.shape[1] > 1:
    y_prob = y_prob[:, 1]

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
final_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC (AUC = {final_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('Taxa de Falsos Positivos')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos')
axes[0].set_title('Curva ROC')
axes[0].legend()
axes[0].grid(True)

# ── Matriz de Confusão (threshold = 0.5) ─────────────────────────────────────
y_pred = (y_prob >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Matriz de Confusão (threshold=0.5)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Relatório de Classificação ---')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


In [ ]:
from sklearn.metrics import classification_report

# ── Busca do threshold ótimo pelo índice de Youden ────────────────────────────
j_scores = tpr - fpr
best_idx  = np.argmax(j_scores)
best_thr  = thresholds[best_idx]
print(f'Threshold ótimo (Youden): {best_thr:.4f}  '
      f'(TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f})')

# aplicar threshold ótimo
y_pred_opt = (y_prob >= best_thr).astype(int)

# classes realmente presentes nos dados
labels_present = np.unique(y_true)

print('\n--- Relatório com threshold ótimo ---')
print(classification_report(
    y_true,
    y_pred_opt,
    labels=labels_present,
    target_names=[CLASS_NAMES[i] for i in labels_present]
))


## 12. Inferência no Conjunto de Teste + Submissão

In [ ]:
# ── TTA (Test-Time Augmentation) ──────────────────────────────────────────────
TTA_STEPS = 5  # número de passagens com augmentation

def get_tta_transforms(img_size: int = IMG_SIZE):
    return A.Compose([
        # Albumentations usa size=(H, W)
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def predict_with_tta(model, df_test, img_dir, tta_steps=TTA_STEPS):
    model.eval()
    all_probs = np.zeros(len(df_test))

    for t in range(tta_steps):
        transform = get_tta_transforms()
        ds = ISICDataset(df_test, img_dir, transform=transform)  # removido is_test
        loader = DataLoader(ds, batch_size=BATCH_SIZE * 2,
                            num_workers=NUM_WORKERS, pin_memory=True)

        step_probs = []
        for batch in tqdm(loader, desc=f'  TTA {t+1}/{tta_steps}'):
            # dependendo da definição do dataset, pode retornar (image, meta) ou (image, meta, label)
            if len(batch) == 2:
                images, meta = batch
            else:
                images, meta, _ = batch

            images = images.to(DEVICE, non_blocking=True)
            meta   = meta.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = model(images, meta)
                probs  = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            step_probs.extend(probs)

        all_probs += np.array(step_probs)

    return all_probs / tta_steps


# corrigir IMG_ID_COL para usar a coluna certa
IMG_ID_COL = 'image_name' if 'image_name' in df_test.columns else 'image'

# rodar predição
test_probs = predict_with_tta(model, df_test, TEST_DIR)

submission = pd.DataFrame({
    IMG_ID_COL: df_test[IMG_ID_COL],
    'target': test_probs
})


In [ ]:
# Distribuição das probabilidades previstas
plt.figure(figsize=(8, 4))
plt.hist(test_probs, bins=50, color='steelblue', edgecolor='white')
plt.axvline(0.5, color='red', linestyle='--', label='Threshold 0.5')
plt.title('Distribuição das Probabilidades Previstas (Teste)')
plt.xlabel('Probabilidade de Malignidade')
plt.ylabel('Contagem')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_prob_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Informações de Hardware e Uso de VRAM

In [ ]:
if DEVICE.type == 'cuda':
    allocated = torch.cuda.max_memory_allocated(0) / 1e9
    reserved  = torch.cuda.max_memory_reserved(0)  / 1e9
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'VRAM alocada  : {allocated:.2f} GB')
    print(f'VRAM reservada: {reserved:.2f} GB')
else:
    print('CPU utilizada – resultados podem ser lentos.')

## 14. Resumo dos Artefatos Gerados

| Arquivo | Descrição |
|---|---|
| `outputs/best_fold0.pth` | Pesos do melhor modelo (fold 0) |
| `outputs/submission.csv` | Probabilidades de malignidade para o conjunto de teste |
| `outputs/eda_overview.png` | Visualização EDA |
| `outputs/learning_curves.png` | Curvas de loss e AUC por época |
| `outputs/evaluation.png` | Curva ROC e matriz de confusão |
| `outputs/test_prob_dist.png` | Distribuição das probabilidades no teste |

### Dicas para melhorar o AUC
- Treinar todos os 5 folds e fazer ensemble das probabilidades  
- Usar `efficientnet_b5` ou `efficientnet_b6` (requer reduzir `BATCH_SIZE`)  
- Aumentar `TTA_STEPS` para 10-20  
- Adicionar metadados tabulares (sexo, idade, localização) ao vetor de features